<a href="https://colab.research.google.com/github/smithblaine014-cyber/irrigation-scheduler/blob/main/irrigation_app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys
!{sys.executable} -m pip install streamlit
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.title("🌱 Irrigation Scheduling App")

st.write("""
Upload a weather dataset containing:

Month
Date
Year
ET_inches
Precipitation_inches
""")

uploaded_file = st.file_uploader("Upload Weather CSV", type=["csv"])

st.sidebar.header("Field Settings")

kc = st.sidebar.number_input("Crop Coefficient (Kc)", value=1.15)
soil_awc = st.sidebar.number_input("Soil Available Water Capacity (in/ft)", value=2.0)
root_depth = st.sidebar.number_input("Root Depth (ft)", value=3.0)
allowed_depletion = st.sidebar.slider("Allowed Depletion (%)", 10, 80, 50)

if uploaded_file:

    df = pd.read_csv(uploaded_file)

    df["Date_full"] = pd.to_datetime({
        "year": df["Year"],
        "month": df["Month"],
        "day": df["Date"]
    })

    eto = df["ET_inches"]
    rain = df["Precipitation_inches"]
    etc = eto * kc

    taw = soil_awc * root_depth
    mad = taw * (allowed_depletion / 100)

    soil_water = taw
    depletion = []

    for i in range(len(df)):

        soil_water = soil_water - etc.iloc[i] + rain.iloc[i]

        if soil_water > taw:
            soil_water = taw

        depletion.append(taw - soil_water)

    df["ETc"] = etc
    df["Depletion"] = depletion
    df["Irrigation_Needed"] = df["Depletion"] > mad

    st.subheader("Results")
    st.write(df)

    st.subheader("Soil Water Depletion")

    fig1, ax1 = plt.subplots()

    ax1.plot(df["Date_full"], df["Depletion"], label="Soil Water Depletion")
    ax1.axhline(mad, linestyle="--", label="Max Allowable Depletion")

    ax1.set_xlabel("Date")
    ax1.set_ylabel("Inches")
    ax1.legend()

    st.pyplot(fig1)

    st.subheader("Daily Water Balance")

    fig2, ax2 = plt.subplots()

    ax2.plot(df["Date_full"], eto, label="Reference ET")
    ax2.plot(df["Date_full"], etc, label="Crop ET")
    ax2.bar(df["Date_full"], rain, label="Rainfall")

    ax2.set_xlabel("Date")
    ax2.set_ylabel("Inches")
    ax2.legend()

    st.pyplot(fig2)

    st.subheader("Recommended Irrigation Days")

    st.write(df[df["Irrigation_Needed"]])

2026-03-11 19:44:49.748 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.749 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.751 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.753 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.753 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 19:44:49.756 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar